# Notebook 02 — Preprocesamiento (encoding + imputación)

Aplica al `consolidado_global.parquet`:

1. Winsorización p1/p99 sobre cols numéricas con outliers extremos.
2. Encoding: OHE para categóricas con cardinalidad < 20, Frequency Encoding ≥ 20.
3. Imputación: mediana + flag `_was_nan` para cada col numérica con NaN.

Estrategia validada en sub-sprint consorcios (mediana + flag ganó por
+0,045 AUC vs KNN/MICE).

**Output**: `data/consolidado_imputado.parquet` listo para modelado.

In [1]:
from pathlib import Path
import warnings, joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

RAIZ = Path('..').resolve()
DATA = RAIZ / 'data'
IMP  = DATA / 'imputers'
IMP.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 150)

df = pd.read_parquet(DATA / 'consolidado_global.parquet')
TARGETS = ['tuvo_atraso', 'tuvo_sobrecosto']
y = df[TARGETS].copy()
X = df.drop(columns=TARGETS).copy()
print(f'X: {X.shape} | y: {y.shape}')

X: (48331, 53) | y: (48331, 2)


## 1. Winsorización p1/p99 sobre numéricas con outliers

In [2]:
cols_num = X.select_dtypes(include=['number']).columns.tolist()
print(f'Numéricas: {len(cols_num)}')

# Winsorizar todas las numéricas continuas (no las binarias 0/1)
winsor_limits = {}
for c in cols_num:
    if X[c].nunique() <= 3:  # binarias o muy discretas → no winsor
        continue
    p1  = X[c].quantile(0.01)
    p99 = X[c].quantile(0.99)
    n_low  = ((X[c] < p1)  & X[c].notna()).sum()
    n_high = ((X[c] > p99) & X[c].notna()).sum()
    X[c] = X[c].clip(lower=p1, upper=p99)
    winsor_limits[c] = (float(p1), float(p99))
    if n_low + n_high > 0:
        print(f'  {c:42s} clip[{p1:.2e}, {p99:.2e}] low={n_low} high={n_high}')

joblib.dump(winsor_limits, IMP / 'winsor_limits.pkl')
print(f'\nCols winsorizadas: {len(winsor_limits)}')

Numéricas: 12
  valor_del_contrato                         clip[0.00e+00, 4.32e+10] low=0 high=484
  valor_de_pago_adelantado                   clip[0.00e+00, 3.22e+08] low=0 high=484
  duracion_planificada_dias                  clip[4.00e+00, 1.36e+03] low=415 high=421
  rup_idx_endeudamiento                      clip[1.50e-03, 1.00e+00] low=157 high=119
  rup_idx_liquidez                           clip[5.62e-01, 1.10e+03] low=172 high=175
  rup_ingresos                               clip[0.00e+00, 1.40e+11] low=0 high=194
  rup_utilidad_neta                          clip[-5.35e+08, 8.65e+09] low=189 high=155
  rup_multas                                 clip[0.00e+00, 1.00e+00] low=0 high=50
  log_valor_contrato                         clip[0.00e+00, 1.06e+01] low=0 high=484
  rup_sanciones                              clip[0.00e+00, 1.00e+00] low=0 high=126
  anios_empresa                              clip[1.50e-01, 1.01e+01] low=310 high=319
  n_proponentes_por_proceso              

## 2. Encoding categóricas (OHE < 20, Freq ≥ 20)

In [3]:
CARD_LIMITE = 20

cols_cat = X.select_dtypes(include=['object']).columns.tolist()
card = X[cols_cat].nunique()
cols_ohe  = card[card <  CARD_LIMITE].index.tolist()
cols_freq = card[card >= CARD_LIMITE].index.tolist()

print(f'Categóricas total: {len(cols_cat)}')
print(f'  OHE  ({len(cols_ohe)}): {cols_ohe}')
print(f'  Freq ({len(cols_freq)}): {cols_freq}')

Categóricas total: 41
  OHE  (26): ['orden', 'rama', 'entidad_centralizada', 'modalidad_de_contratacion', 'condiciones_de_entrega', 'es_grupo', 'es_pyme', 'habilita_pago_adelantado', 'obligaci_n_ambiental', 'origen_de_los_recursos', 'destino_gasto', 'estado_bpin', 'g_nero_representante_legal', 'tipo_de_cuenta', 'el_contrato_puede_ser_prorrogado', 'documentos_tipo', 'nacionalidad_representante_legal', 'obligaciones_postconsumo', 'espostconflicto', 'pilares_del_acuerdo', 'puntos_del_acuerdo', 'tipodocproveedor', 'esta_activa', 'pais_proveedor', 'rup_tamano', 'rup_inhabilidad']
  Freq (15): ['departamento', 'ciudad', 'sector', 'presupuesto_general_de_la_nacion_pgn', 'sistema_general_de_participaciones', 'sistema_general_de_regal_as', 'recursos_propios_alcald_as_gobernaciones_y_resguardos_ind_genas_', 'recursos_de_credito', 'recursos_propios', 'codigo_de_categoria_principal', 'localizaci_n', 'departamento_proveedor', 'rup_empleados', 'rup_activo_total', 'rup_patrimonio']


In [4]:
# OHE
# Tratar NaN como categoría 'desconocido' para no romper encoding
for c in cols_ohe:
    X[c] = X[c].fillna('desconocido')
X_ohe = pd.get_dummies(X[cols_ohe], drop_first=False, dtype=np.int8)
print(f'OHE genera: {X_ohe.shape[1]} cols')

OHE genera: 116 cols


In [5]:
# Frequency Encoding
freq_maps = {}
X_freq = pd.DataFrame(index=X.index)
for c in cols_freq:
    # NaN → frecuencia 0 (señal de 'desconocido')
    fmap = X[c].value_counts(normalize=True).to_dict()
    freq_maps[c] = fmap
    X_freq[c + '_freq'] = X[c].map(fmap).fillna(0.0).astype(np.float32)
joblib.dump(freq_maps, IMP / 'freq_maps.pkl')
print(f'Freq genera: {X_freq.shape[1]} cols')

Freq genera: 15 cols


In [6]:
# Numéricas (incluye NaN — imputaremos en sig paso)
X_num = X[cols_num].astype(np.float32)

X_full = pd.concat([X_num, X_ohe, X_freq], axis=1)
print(f'X_full preliminar: {X_full.shape}')

X_full preliminar: (48331, 143)


## 3. Imputación: mediana + flag `_was_nan`

In [7]:
cols_con_nan = X_full.columns[X_full.isna().any()].tolist()
print(f'Cols con NaN ({len(cols_con_nan)}):')
for c in cols_con_nan:
    print(f'  {c:42s}  {X_full[c].isna().mean()*100:5.1f}% NaN')

Cols con NaN (9):
  duracion_planificada_dias                    12.6% NaN
  rup_idx_endeudamiento                        63.5% NaN
  rup_idx_liquidez                             63.9% NaN
  rup_ingresos                                 60.1% NaN
  rup_utilidad_neta                            60.0% NaN
  rup_multas                                   59.7% NaN
  rup_sanciones                                51.0% NaN
  anios_empresa                                33.5% NaN
  n_proponentes_por_proceso                    51.8% NaN


In [8]:
# Crear flags _was_nan ANTES de imputar
flags = X_full[cols_con_nan].isna().astype(np.int8).add_suffix('_was_nan')

# Imputar con mediana
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='median')
X_imp = imp.fit_transform(X_full.values)
X_imp = pd.DataFrame(X_imp, columns=X_full.columns, index=X_full.index)

# Concatenar flags
X_final = pd.concat([X_imp, flags], axis=1)
joblib.dump(imp, IMP / 'imputer_mediana.pkl')

print(f'X_final: {X_final.shape}')
print(f'NaN restantes: {X_final.isna().sum().sum()}')

X_final: (48331, 152)
NaN restantes: 0


In [9]:
# Concatenar targets y exportar
final = X_final.copy()
for t in TARGETS:
    final[t] = y[t].values
final.to_parquet(DATA / 'consolidado_imputado.parquet', index=False)
print(f'Exportado: consolidado_imputado.parquet {final.shape}')
print(f'Features: {final.shape[1] - 2}, targets: 2')

Exportado: consolidado_imputado.parquet (48331, 154)
Features: 152, targets: 2


In [10]:
# Resumen
resumen = {
    'n_filas':              len(final),
    'n_features_finales':   final.shape[1] - 2,
    'n_cols_winsorizadas':  len(winsor_limits),
    'n_cols_ohe':           len(cols_ohe),
    'n_cols_freq':          len(cols_freq),
    'n_flags_was_nan':      flags.shape[1],
    'cols_ohe_generadas':   X_ohe.shape[1],
    'tasa_atraso':          round(y['tuvo_atraso'].mean(), 4),
    'tasa_sobrecosto':      round(y['tuvo_sobrecosto'].mean(), 4),
}
for k, v in resumen.items(): print(f'  {k:30s} {v}')

  n_filas                        48331
  n_features_finales             152
  n_cols_winsorizadas            12
  n_cols_ohe                     26
  n_cols_freq                    15
  n_flags_was_nan                9
  cols_ohe_generadas             116
  tasa_atraso                    0.6048
  tasa_sobrecosto                0.8341
